# vLLM Benchmark Results Analysis
Visualize output from `run_benchmark.py`: Cold / Warm / Long Context phases.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "SimHei"],
    "axes.unicode_minus": False,
    "figure.dpi": 120,
})

# Load the latest benchmark result CSV
result_files = sorted(glob.glob("results/benchmark_*.csv"))
if not result_files:
    raise FileNotFoundError("No benchmark result CSV found in results/. Run run_benchmark.py first.")

csv_path = result_files[-1]
print(f"Loading: {csv_path}")
df = pd.read_csv(csv_path)
df.head(10)


In [ ]:
# Phase A: Cold Start
cold = df[df["phase"] == "cold"]
if not cold.empty:
    r = cold.iloc[0]
    print("=== Phase A: Cold Start ===")
    print(f"  TTFT         : {r['ttft_ms']:.1f} ms")
    print(f"  Total latency: {r['latency_ms']:.1f} ms")
    print(f"  Decode TPS   : {r['decode_tps']:.1f} tokens/s")
    print(f"  GPU mem peak : {r['gpu_mem_peak_mb']:.0f} MB")

In [ ]:
# Phase B: Warm Start — TTFT & TPS across runs
warm = df[df["phase"] == "warm"].copy()
if not warm.empty:
    warm["run"] = range(1, len(warm) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.bar(warm["run"], warm["ttft_ms"], color="#4C78A8")
    ax1.set_xlabel("Run #")
    ax1.set_ylabel("TTFT (ms)")
    ax1.set_title("Phase B: TTFT per run")
    ax1.axvline(x=2.5, color="red", linestyle="--", alpha=0.5, label="eval boundary")
    ax1.legend()

    ax2.bar(warm["run"], warm["decode_tps"], color="#F58518")
    ax2.set_xlabel("Run #")
    ax2.set_ylabel("Decode TPS")
    ax2.set_title("Phase B: Decode TPS per run")
    ax2.axvline(x=2.5, color="red", linestyle="--", alpha=0.5, label="eval boundary")
    ax2.legend()

    plt.tight_layout()
    plt.savefig("results/phase_b_warm.png", bbox_inches="tight")
    plt.show()

    # P50/P95 for eval runs (3-5)
    eval_runs = warm[warm["run"] >= 3]
    print("=== Phase B: Eval Runs (3-5) ===")
    print(f"  TTFT  p50={eval_runs['ttft_ms'].quantile(0.5):.1f}ms  p95={eval_runs['ttft_ms'].quantile(0.95):.1f}ms")
    print(f"  TPS   p50={eval_runs['decode_tps'].quantile(0.5):.1f}  p95={eval_runs['decode_tps'].quantile(0.95):.1f}")

In [ ]:
# Phase C: Long Context — TTFT & GPU memory vs input length
lc = df[df["phase"] == "long_context"].copy()
if not lc.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(lc["input_tokens"], lc["ttft_ms"], marker="o", color="#4C78A8", linewidth=2)
    ax1.set_xlabel("Input tokens")
    ax1.set_ylabel("TTFT (ms)")
    ax1.set_title("Phase C: TTFT vs Input Length")
    ax1.grid(True, alpha=0.3)

    ax2.plot(lc["input_tokens"], lc["gpu_mem_peak_mb"], marker="s", color="#E45756", linewidth=2)
    ax2.set_xlabel("Input tokens")
    ax2.set_ylabel("GPU Memory Peak (MB)")
    ax2.set_title("Phase C: GPU Memory vs Input Length")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("results/phase_c_long_context.png", bbox_inches="tight")
    plt.show()

    print("=== Phase C: Long Context Details ===")
    print(lc[["label", "input_tokens", "ttft_ms", "decode_tps", "gpu_mem_peak_mb"]].to_string(index=False))

In [ ]:
# Full summary table
print("=== Full Benchmark Results ===")
print(df[["phase", "label", "input_tokens", "output_tokens", "ttft_ms", "decode_tps", "gpu_mem_peak_mb", "success"]].to_string(index=False))